In [6]:
import pandas as pd

# 1. Load original CSV
df_raw = pd.read_csv("kidney_disease.csv")

# 2. Work on a copy so the original stays untouched
df = df_raw.copy()

# 3. Standardize column names
df.columns = df.columns.str.strip().str.lower()

# 4. Replace placeholder missing values with proper NaN
df = df.replace(["?", "", " "], pd.NA)

# 5. Strip whitespace from all object columns (gets rid of '\t', leading/trailing spaces)
obj_cols = df.select_dtypes("object").columns
df[obj_cols] = df[obj_cols].apply(lambda col: col.str.strip())

# 6. Fix messy category labels
df["dm"] = df["dm"].replace({"yes": "yes", "no": "no",
                             "\tyes": "yes", "\tno": "no", " yes": "yes"})
df["cad"] = df["cad"].replace({"no": "no", "yes": "yes", "\tno": "no"})
df["classification"] = df["classification"].replace({"ckd\t": "ckd"})

# 7. Convert numeric-like columns from strings to proper numeric
numeric_like = [
    "age", "bp", "sg", "al", "su",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wc", "rc"
]

for col in numeric_like:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 8. Mark categorical columns (optional but helpful later)
cat_cols = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad",
    "appet", "pe", "ane",
    "classification"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

# 9. Drop rows with any missing values to get a complete-case dataset
df_clean = df.dropna().reset_index(drop=True)

print("Original shape:", df_raw.shape)
print("Cleaned shape:", df_clean.shape)

# 10. Save cleaned data to a new CSV
df_clean.to_csv("kidney_disease_clean.csv", index=False)
print("Cleaned dataset saved as 'kidney_disease_clean.csv'")


Original shape: (400, 26)
Cleaned shape: (158, 26)
Cleaned dataset saved as 'kidney_disease_clean.csv'


In [7]:
df_clean = pd.read_csv("kidney_disease_clean.csv")

# Identify categorical columns
cat_cols = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad",
    "appet", "pe", "ane",
    "classification"
]

# Ensure they're categories
for col in cat_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("category")

# Create dummy variables (drop_first avoids multicollinearity)
df_model_ready = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

print("New shape after dummy encoding:", df_model_ready.shape)

# Save final model-ready CSV
df_model_ready.to_csv("kidney_disease_model_ready.csv", index=False)
print("Saved: kidney_disease_model_ready.csv")


New shape after dummy encoding: (158, 26)
Saved: kidney_disease_model_ready.csv


In [8]:
df.dtypes


id                   int64
age                float64
bp                 float64
sg                 float64
al                 float64
su                 float64
rbc               category
pc                category
pcc               category
ba                category
bgr                float64
bu                 float64
sc                 float64
sod                float64
pot                float64
hemo               float64
pcv                float64
wc                 float64
rc                 float64
htn               category
dm                category
cad               category
appet             category
pe                category
ane               category
classification    category
dtype: object

In [9]:

df = pd.read_csv("kidney_disease_model_ready.csv")

print("Before fix dtypes:")
print(df.dtypes)

# 2. Columns that are currently bool and should be 0/1 integers
bool_cols = [
    "rbc_normal", "pc_normal", "pcc_present", "ba_present",
    "htn_yes", "dm_yes", "cad_yes",
    "appet_poor", "pe_yes", "ane_yes",
    "classification_notckd"
]

# 3. Convert those to integers (0/1), but only if they exist
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

# 4. (Optional but nice) make sure all non-id columns are numeric where possible
for col in df.columns:
    if col not in ["id"]:
        df[col] = pd.to_numeric(df[col], errors="ignore")

print("\nAfter fix dtypes:")
print(df.dtypes)

# 5. Save back to the SAME CSV (overwrites old model-ready version)
df.to_csv("kidney_disease_model_ready.csv", index=False)
print("\nUpdated 'kidney_disease_model_ready.csv' saved.")


Before fix dtypes:
id                         int64
age                      float64
bp                       float64
sg                       float64
al                       float64
su                       float64
bgr                      float64
bu                       float64
sc                       float64
sod                      float64
pot                      float64
hemo                     float64
pcv                      float64
wc                       float64
rc                       float64
rbc_normal                  bool
pc_normal                   bool
pcc_present                 bool
ba_present                  bool
htn_yes                     bool
dm_yes                      bool
cad_yes                     bool
appet_poor                  bool
pe_yes                      bool
ane_yes                     bool
classification_notckd       bool
dtype: object

After fix dtypes:
id                         int64
age                      float64
bp                      

C:\Users\Koolt\AppData\Local\Temp\ipykernel_2964\1302502566.py:22: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
